In [2]:
# standardní knihovny Pythonu
import sys
import time
import re
from pathlib import Path

# knihovny třetích stran
import pandas as pd
import requests
from bs4 import BeautifulSoup

# nastavení root složky projektu pro import utility modulů
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# sdílené konstanty a společné schéma datasetu
from utils.constants import HEADERS
from utils.schema import FINAL_COLUMNS

# základní technické čištění textu
from utils.text_cleaning import (
    clean_base_text,
    join_text_parts
)

In [7]:
# načtení seznamu kódů předmětů
BI_COURSE_CODES_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "bi_course_codes.csv"
)

df_course_codes = pd.read_csv(
    BI_COURSE_CODES_PATH,
    sep=";"
)

# kódy předmětů ze sekce Programme Information
programmeinfo_course_codes = df_course_codes.loc[
    df_course_codes["source_group"] == "programmeinfo",
    "course_code"
].tolist()

# kódy předmětů ze sekce TEM
tem_course_codes = df_course_codes.loc[
    df_course_codes["source_group"] == "tem",
    "course_code"
].tolist()

In [20]:
# převod HTML obsahu na čistý text
def html_to_text(value):

    if not value:
        return pd.NA

    text = BeautifulSoup(
        value,
        "html.parser"
    ).get_text(" ", strip=True)

    return text if text else None

In [9]:
# extrakce textu z pole podle CSS třídy
def get_field(soup, class_name):

    div = soup.find(
        "div",
        class_=class_name
    )

    if not div:
        return pd.NA

    label = div.find(
        "div",
        class_="label"
    )

    if label:
        label.decompose()

    return clean_base_text(
        div.get_text(" ", strip=True)
    )


# extrakce názvu předmětu
def get_title(soup):

    title_tag = soup.select_one(
        "div.node-cd > h2:not(.element-invisible)"
    )

    if not title_tag:
        return pd.NA

    text = title_tag.get_text(" ", strip=True)
    text = re.sub(r"^[A-Z]+ \d+\s+", "", text)

    return clean_base_text(text)


# technické odstranění nadpisu z pole hodnocení
def clean_assessment_text(value):

    if pd.isna(value):
        return pd.NA

    text = str(value)
    text = re.sub(r"^Assessments\s*", "", text, flags=re.IGNORECASE)

    return clean_base_text(text)


# normalizace semestru do společných kategorií
def normalize_semester(semester):

    if pd.isna(semester):
        return pd.NA

    semester_lower = str(semester).lower()

    if any(x in semester_lower for x in ["spring", "vår"]):
        return "summer"

    if any(x in semester_lower for x in ["autumn", "fall", "høst"]):
        return "winter"

    return pd.NA


# extrakce názvu semestru ze zdrojové hodnoty BI
def extract_bi_semester_name(semester):

    if pd.isna(semester):
        return pd.NA

    semester_str = str(semester).strip()
    semester_str = re.sub(r"^\d{4}[-\s]*", "", semester_str)

    return clean_base_text(semester_str)


# extrakce akademického roku ze zdrojové hodnoty semestru
def extract_academic_year(semester):

    if pd.isna(semester):
        return pd.NA

    match = re.search(
        r"(20\d{2})",
        str(semester)
    )

    return match.group(1) if match else pd.NA

In [10]:
# stažení a naparsování sylabu ze zdroje Programme Information
def stahni_sylabus_programmeinfo(course_code, semester="2026-autumn"):

    url = f"https://programmeinfo.bi.no/nb/course/{course_code}/{semester}"

    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=15
        )

        if response.status_code != 200:
            print(f"CHYBA {response.status_code} pro {course_code}")
            return None

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        semester_value = get_field(
            soup,
            "field-semesters-refs"
        )

        credits = get_field(
            soup,
            "field-credits"
        )

        learning_outcomes = join_text_parts([
            get_field(soup, "field-learning-outcome-knowledge"),
            get_field(soup, "field-learning-outcome-skills"),
            get_field(soup, "field-learning-outcome-reflec")
        ])

        return {
            "course_title": get_title(soup),
            "course_code": get_field(soup, "field-course-codes-ref") or course_code,
            "university": "BI Norwegian Business School",
            "country": "Norway",
            "source_url": url,
            "academic_year": extract_academic_year(semester_value),
            "semester": extract_bi_semester_name(semester_value),
            "semester_normalized": normalize_semester(semester_value),
            "study_level": get_field(soup, "field-study-levels-ref"),
            "credits": credits,
            "credit_system": "ECTS",
            "ects_credits": credits,
            "aims": get_field(soup, "field-introduction"),
            "course_content": get_field(soup, "field-course-content"),
            "learning_outcomes": learning_outcomes,
            "assessment_methods": clean_assessment_text(
                get_field(soup, "field-assessments-row")
            ),
            "literature": get_field(soup, "field-reading-list-citations")
        }

    except requests.RequestException as e:
        print(f"CHYBA při stahování {course_code}: {e}")
        return None

In [11]:
# převod interního označení období na název semestru
def term_to_semester(term):

    if term == "202620":
        return "Fall"

    if term == "202610":
        return "Spring"

    return str(term)


# extrakce akademického roku z interního kódu období
def term_to_academic_year(term):

    if pd.isna(term):
        return pd.NA

    return str(term)[:4]


# odstranění technického upozornění z úvodu předmětu
def remove_preliminary_notice(text):

    if pd.isna(text):
        return pd.NA

    text = str(text).strip()

    text = re.sub(
        r"^Please note that this is a preliminary course description\. The final version will be published in June 2026\.\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    return clean_base_text(text)

In [13]:
# stažení a naparsování sylabu z portal.bi.no
def stahni_sylabus_tem(
    course_code,
    term="202620",
    language="en"
):

    subject, number = course_code.split("-")

    api_url = (
        f"https://portal.bi.no/api/course-catalog/"
        f"course-details/{subject}/{number}/{term}/{language}"
    )

    page_url = (
        f"https://portal.bi.no/kurs/#/course/"
        f"en/{subject}/{number}/{term}"
    )

    try:

        response = requests.get(
            api_url,
            headers=HEADERS,
            timeout=15
        )

        if response.status_code != 200:
            print(f"CHYBA {response.status_code} pro {course_code}")
            return None

        data = response.json()

        semester_value = term_to_semester(
            data.get("semester")
        )

        credits = data.get("credits")

        learning_outcomes = join_text_parts([
            html_to_text(data.get("learningOutcomeKnowledge")),
            html_to_text(data.get("learningOutcomeSkills")),
            html_to_text(data.get("learningOutcomeReflection"))
        ])

        return {
            "course_title": clean_base_text(data.get("title")),
            "course_code": (
                f"{data.get('subjectCode')} "
                f"{data.get('courseNumber')}"
            ),
            "university": "BI Norwegian Business School",
            "country": "Norway",
            "source_url": page_url,
            "academic_year": term_to_academic_year(
                data.get("semester")
            ),
            "semester": semester_value,
            "semester_normalized": normalize_semester(
                semester_value
            ),
            "study_level": clean_base_text(
                data.get("levelOfStudy")
            ),
            "credits": credits,
            "credit_system": "ECTS",
            "ects_credits": credits,
            "aims": remove_preliminary_notice(
                html_to_text(data.get("introduction"))
            ),
            "course_content": html_to_text(
                data.get("courseContent")
            ),
            "learning_outcomes": learning_outcomes,
            "assessment_methods": clean_base_text(
                data.get("examType")
            ),
            "literature": html_to_text(
                data.get("readingList")
            )
        }

    except requests.RequestException as e:

        print(f"CHYBA při stahování {course_code}: {e}")

        return None

In [14]:
# stažení sylabů ze zdroje Programme Information
# crawl-delay 10 sekund odpovídá pravidlům uvedeným v robots.txt
programmeinfo_rows = []

for course_code in programmeinfo_course_codes:

    print(f"Stahuji Programme Information kurz {course_code}...")

    row = stahni_sylabus_programmeinfo(course_code)

    if row is not None:
        programmeinfo_rows.append(row)

    time.sleep(10)

df_programmeinfo = pd.DataFrame(programmeinfo_rows)

print(f"Staženo Programme Information kurzů: {len(df_programmeinfo)}")

Stahuji Programme Information kurz BST-2413...
Stahuji Programme Information kurz EBA-1180...
Stahuji Programme Information kurz EBA-3400...
Stahuji Programme Information kurz EBA-3501...
Stahuji Programme Information kurz EBA-3610...
Stahuji Programme Information kurz EBA-3630...
Stahuji Programme Information kurz EDI-3400...
Stahuji Programme Information kurz EDI-3410...
Stahuji Programme Information kurz EDI-3500...
Stahuji Programme Information kurz EDI-3510...
Stahuji Programme Information kurz EDI-3520...
Stahuji Programme Information kurz EDI-3530...
Stahuji Programme Information kurz ELE-3709...
Stahuji Programme Information kurz ELE-3732...
Stahuji Programme Information kurz ELE-3736...
Stahuji Programme Information kurz ELE-3750...
Stahuji Programme Information kurz ELE-3781...
Stahuji Programme Information kurz ELE-3795...
Stahuji Programme Information kurz ELE-3798...
Stahuji Programme Information kurz ELE-3905...
Stahuji Programme Information kurz ELE-3908...
Stahuji Progr

In [15]:
# stažení sylabů ze zdroje TEM
tem_rows = []

for course_code in tem_course_codes:

    print(f"Stahuji TEM kurz {course_code}...")

    row = stahni_sylabus_tem(course_code)

    if row is not None:
        tem_rows.append(row)

    # omezení frekvence requestů vůči API
    time.sleep(5)

df_tem = pd.DataFrame(tem_rows)

print(f"Staženo TEM kurzů: {len(df_tem)}")

Stahuji TEM kurz TEM-0043...
Stahuji TEM kurz TEM-0057...
Stahuji TEM kurz TEM-0042...
Stahuji TEM kurz TEM-0421...
Stahuji TEM kurz TEM-0045...
Stahuji TEM kurz TEM-0480...
Stahuji TEM kurz TEM-0420...
Stahuji TEM kurz TEM-0481...
Stahuji TEM kurz TEM-0241...
Stahuji TEM kurz TEM-0240...
Stahuji TEM kurz TEM-0055...
Stahuji TEM kurz TEM-0603...
Stahuji TEM kurz TEM-0460...
Stahuji TEM kurz TEM-0037...
Stahuji TEM kurz TEM-0602...
Stahuji TEM kurz TEM-0601...
Stahuji TEM kurz TEM-0640...
Stahuji TEM kurz TEM-0029...
Stahuji TEM kurz TEM-0035...


C:\Users\semlu\AppData\Local\Temp\ipykernel_21152\3367685684.py:7: MarkupResemblesLocatorWarning: The input looks more like a URL than markup. You may want to use an HTTP client like requests to get the document behind the URL, and feed that document to Beautiful Soup.
  text = BeautifulSoup(


Stahuji TEM kurz TEM-0203...
Stahuji TEM kurz TEM-0521...
Stahuji TEM kurz TEM-0202...
Stahuji TEM kurz TEM-0520...
Stahuji TEM kurz TEM-0204...
Stahuji TEM kurz TEM-0221...
Stahuji TEM kurz TEM-0220...
Stahuji TEM kurz TEM-0222...
Stahuji TEM kurz TEM-0056...
Stahuji TEM kurz TEM-0400...
Stahuji TEM kurz TEM-0441...
Stahuji TEM kurz TEM-0600...
Stahuji TEM kurz TEM-0440...
Stahuji TEM kurz TEM-0401...
Stahuji TEM kurz TEM-0641...
Stahuji TEM kurz TEM-0541...
Stahuji TEM kurz TEM-0252...
Stahuji TEM kurz TEM-0540...
Stahuji TEM kurz TEM-0251...
Staženo TEM kurzů: 38


In [16]:
# spojení dat z obou zdrojů BI
df_bi = pd.concat(
    [df_programmeinfo, df_tem],
    ignore_index=True
)

# sjednocení pořadí sloupců podle společného schématu
df_bi = df_bi[FINAL_COLUMNS]

In [19]:
# export naparsovaných BI sylabů do interim vrstvy
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "bi_syllabi_raw_parsed.csv"
)

df_bi.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Uloženo {len(df_bi)} kurzů do {OUTPUT_PATH.name}")

Uloženo 127 kurzů do bi_syllabi_raw_parsed.csv
